# Data Structures, Algorithms & Complexity — Hands-On

**Software Engineering · Week 03**

Offline notebook: stdlib-only examples of lists, maps, traversal, search, sorting, complexity, and tiny benchmarks.

## 0. Setup: small data and timing helper

In [ ]:
from collections import deque, Counter
from bisect import bisect_left
from time import perf_counter
import random

random.seed(0)
rows = [(f"u{i}", f"name-{i}") for i in range(200)]
lookups = [f"u{random.randrange(200)}" for _ in range(80)]

def bench(label, fn, repeat=200):
    start = perf_counter()
    for _ in range(repeat):
        fn()
    elapsed_ms = (perf_counter() - start) * 1000
    print(f"{label:<28} {elapsed_ms:7.3f} ms")

## 1. Array/list scan vs hash-map lookup

In [ ]:
def scan_all():
    total = 0
    for key in lookups:
        for uid, _ in rows:
            if uid == key:
                total += 1
                break
    return total

index = dict(rows)
def map_all():
    return sum(1 for key in lookups if key in index)

bench("repeated linear scans", scan_all)
bench("hash-map index lookups", map_all)
print("same result:", scan_all() == map_all())

## 2. Sorting enables binary search

In [ ]:
sorted_rows = sorted(rows)
ids = [uid for uid, _ in sorted_rows]

def binary_find(key):
    i = bisect_left(ids, key)
    return i < len(ids) and ids[i] == key

def binary_all():
    return sum(1 for key in lookups if binary_find(key))

bench("binary search sorted list", binary_all)
print("binary search requires sorted ids:", ids[:4])

## 3. Stack vs queue: DFS and BFS

In [ ]:
graph = {"A":["B","C"], "B":["D","E"], "C":["F"], "D":[], "E":["F"], "F":[]}

def bfs(start):
    seen, order, q = {start}, [], deque([start])
    while q:
        node = q.popleft(); order.append(node)
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt); q.append(nxt)
    return order

def dfs(start):
    seen, order, stack = set(), [], [start]
    while stack:
        node = stack.pop()
        if node in seen: continue
        seen.add(node); order.append(node)
        stack.extend(reversed(graph[node]))
    return order

print("BFS:", bfs("A"))
print("DFS:", dfs("A"))

## 4. Graph cycles require a visited set

In [ ]:
cyclic = {"A":["B"], "B":["C"], "C":["A"]}

def reachable(start):
    seen, q = {start}, deque([start])
    while q:
        node = q.popleft()
        for nxt in cyclic[node]:
            if nxt not in seen:
                seen.add(nxt); q.append(nxt)
    return seen

print("reachable despite cycle:", reachable("A"))

## 5. Counting with a map instead of nested comparisons

In [ ]:
events = ["login", "search", "login", "checkout", "search", "search"]
counts = Counter(events)
print(counts)
print("top event:", counts.most_common(1)[0])

## 6. Memory-vs-speed: an index duplicates key storage

In [ ]:
print("row count:", len(rows), "index entries:", len(index))
print("sample index lookup:", index["u42"])
print("tradeoff: extra structure kept in memory to avoid repeated scans")

## 7. Tiny input benchmark sanity check

In [ ]:
tiny_rows = rows[:5]
tiny_lookups = ["u1", "u4", "u0"]

def tiny_scan():
    return [next(name for uid, name in tiny_rows if uid == key) for key in tiny_lookups]

def tiny_map():
    tiny_index = dict(tiny_rows)
    return [tiny_index[key] for key in tiny_lookups]

bench("tiny scan incl no index", tiny_scan, repeat=1000)
bench("tiny build map + lookup", tiny_map, repeat=1000)
print("constants matter on tiny inputs")

## Exercises
1. Add a heap-based top-k example using `heapq`.
2. Change the graph to include weighted edges; why is BFS no longer enough?
3. Benchmark repeated lookups as `len(rows)` grows from 10 to 10_000.
4. Add a stale index bug: mutate `rows` without updating `index`.

## Links
- Literature note: `02 Literature Notes/Software Engineering/Data Structures, Algorithms & Complexity`
- Snippets: `04 Code Snippets/Software Engineering/SE Week 03 Graph BFS DFS Traversal`, `.../SE Week 03 Binary Search Sort and Hash Index`
- MOC: `06 Maps of Content/Software Engineering Concepts`